# Thử train từng bài từ CSV 33 điểm khớp

Upload `bodypoint_pose_landmarks.csv` do `export_pose_landmarks.ipynb` tạo. Notebook xuất một model TFLite và một file nhãn cho **mỗi bài có đủ hai nhãn**. Các bài chỉ có một ảnh gốc mỗi nhãn vẫn được train để kiểm tra pipeline, nhưng **không có phép đo độ chính xác hợp lệ**; cần bổ sung nhiều ảnh độc lập trước khi dùng chấm tư thế thật. Các model ngoài `neck` chưa được ứng dụng Android nạp tự động.


In [ ]:
!pip -q install tensorflow scikit-learn
import csv, json, pathlib, random
import numpy as np
import tensorflow as tf
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
LABELS = ['Dung', 'Sai']
print('TensorFlow:', tf.__version__)


In [ ]:
print('Chọn bodypoint_pose_landmarks.csv do export_pose_landmarks.ipynb tạo.')
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if len(csv_names) != 1 or len(uploaded) != 1:
    raise ValueError('Hãy tải lên đúng một file CSV dữ liệu điểm khớp.')
columns = [f'p{i}_{axis}' for i in range(33) for axis in ('x', 'y', 'z', 'visibility')]
with open(csv_names[0], newline='', encoding='utf-8-sig') as stream:
    reader = csv.DictReader(stream)
    required = {'exercise', 'label', 'source_image', *columns}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError('CSV thiếu cột: ' + ', '.join(sorted(missing)))
    records = list(reader)
print('Tổng số dòng:', len(records))
print('Theo bài và nhãn:', Counter((r['exercise'], r['label']) for r in records))


In [ ]:
# Chuẩn hóa 33 x (x, y, z, visibility), giống NeckModel.java trong Android.
def features(a):
    a = np.asarray(a, dtype=np.float32).reshape(33, 4).copy()
    if not np.isfinite(a).all():
        return None
    scale = np.linalg.norm(a[11, :2] - a[12, :2])
    if scale < 0.02 or min(a[0, 3], a[11, 3], a[12, 3]) < 0.35 or max(a[1, 3], a[4, 3], a[7, 3], a[8, 3]) < 0.35:
        return None
    center = (a[11, :3] + a[12, :3]) / 2
    a[:, :3] = (a[:, :3] - center) / scale
    return a.flatten()

EXERCISES = ['neck', 'shoulder', 'biceps', 'wrist_up', 'wrist_down']
groups = {exercise: {'X': [], 'y': [], 'used': []} for exercise in EXERCISES}
for record in records:
    exercise = record['exercise'].strip().lower()
    if exercise not in groups:
        continue
    label = record['label'].strip()
    if label not in LABELS:
        raise ValueError('Nhãn không hợp lệ: ' + label)
    try:
        values = [float(record[name]) for name in columns]
    except (ValueError, TypeError) as exc:
        raise ValueError('Tọa độ lỗi ở ảnh: ' + record['source_image']) from exc
    row = features(values)
    if row is None:
        print('Bỏ qua điểm khớp không đủ rõ:', record['source_image'])
        continue
    groups[exercise]['X'].append(row)
    groups[exercise]['y'].append(LABELS.index(label))
    groups[exercise]['used'].append(record['source_image'])
for exercise, group in groups.items():
    group['X'] = np.asarray(group['X'], dtype=np.float32).reshape(-1, 132)
    group['y'] = np.asarray(group['y'], dtype=np.int32)
    counts = np.bincount(group['y'], minlength=2)
    print(exercise, 'mẫu gốc:', dict(zip(LABELS, counts.tolist())))
    if min(counts) < 1:
        raise ValueError(f'{exercise}: cần ít nhất một ảnh gốc cho mỗi nhãn Dung/Sai')


In [ ]:
# Chỉ bài có >=10 ảnh gốc mỗi nhãn mới chia được tập test nhỏ.
# Lật landmark của tập train; không tính ảnh lật là mẫu kiểm tra độc lập.
MIRROR_PAIRS = [(1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
                (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)]
def mirror_features(rows):
    mirrored = rows.reshape(-1, 33, 4).copy()
    mirrored[:, :, 0] *= -1
    for left, right in MIRROR_PAIRS:
        pair = mirrored[:, [left, right], :].copy()
        mirrored[:, left, :] = pair[:, 1, :]
        mirrored[:, right, :] = pair[:, 0, :]
    return mirrored.reshape(-1, 132)

trained = {}
manifest = []
for exercise, group in groups.items():
    X, y, used = group['X'], group['y'], group['used']
    counts = np.bincount(y, minlength=2)
    if min(counts) >= 10:
        train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.2,
                                               random_state=42, stratify=y)
        train_idx, val_idx = train_test_split(train_idx, test_size=0.25,
                                             random_state=42, stratify=y[train_idx])
    else:
        train_idx = np.arange(len(y))
        val_idx = test_idx = np.array([], dtype=np.int32)
        print(exercise, ': CHỈ THỬ PIPELINE — quá ít ảnh gốc để đo độ chính xác')
    for split, ids in [('train', train_idx), ('validation', val_idx), ('test', test_idx)]:
        manifest.extend((exercise, split, LABELS[int(y[i])], used[int(i)]) for i in ids)
    X_fit = np.concatenate([X[train_idx], mirror_features(X[train_idx])])
    y_fit = np.concatenate([y[train_idx], y[train_idx]])
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(132,)),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(2, activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    fit_options = {'batch_size': 16, 'epochs': 80, 'verbose': 0}
    if len(val_idx):
        fit_options['validation_data'] = (X[val_idx], y[val_idx])
        fit_options['callbacks'] = [tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=10, restore_best_weights=True)]
    model.fit(X_fit, y_fit, **fit_options)
    accuracy = None
    if len(test_idx):
        pred = model.predict(X[test_idx], verbose=0).argmax(axis=1)
        accuracy = float(np.mean(pred == y[test_idx]))
        print(exercise, 'test accuracy (tập rất nhỏ):', accuracy)
        print(classification_report(y[test_idx], pred, labels=[0, 1],
                                    target_names=LABELS, zero_division=0))
    trained[exercise] = (model, accuracy, counts.tolist())
with open('/content/pose_split_manifest.csv', 'w', newline='', encoding='utf-8') as stream:
    writer = csv.writer(stream)
    writer.writerow(['exercise', 'split', 'label', 'source_image'])
    writer.writerows(manifest)


In [ ]:
import zipfile
bundle = pathlib.Path('/content/bodypoint_pose_models.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for exercise, (model, accuracy, counts) in trained.items():
        converter = tf.lite.TFLiteConverter.from_keras_model(model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        model_path = pathlib.Path(f'/content/{exercise}_stretch.tflite')
        model_path.write_bytes(converter.convert())
        labels_path = pathlib.Path(f'/content/{exercise}_labels.txt')
        labels_path.write_text('\n'.join(LABELS) + '\n', encoding='utf-8')
        interpreter = tf.lite.Interpreter(model_path=str(model_path))
        interpreter.allocate_tensors()
        assert interpreter.get_input_details()[0]['shape'].tolist() == [1, 132]
        assert interpreter.get_output_details()[0]['shape'].tolist() == [1, 2]
        info = {'exercise': exercise, 'labels': LABELS, 'original_counts': dict(zip(LABELS, counts)),
                'test_accuracy': accuracy, 'validated': accuracy is not None,
                'warning': 'Demo only until many independent real photos are collected'}
        info_path = pathlib.Path(f'/content/{exercise}_model_info.json')
        info_path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
        for path in (model_path, labels_path, info_path):
            archive.write(path, path.name)
    archive.write('/content/pose_split_manifest.csv', 'pose_split_manifest.csv')
print('Đã xuất:', bundle)
files.download(str(bundle))
